[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/05-explainability/08-debugging_a_bad_match.ipynb)

In [1]:
# !pip install mbox

# Debugging a Bad Match

Sooner or later, a query that looks like it should obviously work will return nothing, or will return the wrong candidate, or will silently drop a result you expected to see. `01-index_types_and_recall_modes.ipynb` through `05-ident_fields_explained.ipynb` gave you the mechanics: how scores are computed, why matches disappear rather than fade, and how each `IndexType` behaves differently. This notebook is where that knowledge earns its keep, five real, reproducible causes of a bad match, diagnosed the way you would actually diagnose one: by running one more query, not by guessing.

As in the rest of this directory, the data for each scenario is loaded from `datasets/`, not built inline.

In this notebook you will:

1. See a perfect match on one field disappear entirely because of an unrelated field in the same query
2. Learn the diagnostic technique of isolating fields, one at a time, to find which one is causing a failure
3. Recognize when a short field has simply run out of fuzzy tolerance
4. Catch an `IndexType` mismatch changing what "the same query" actually matches
5. Catch a candidate being silently filtered by `minimum_quality`, rather than genuinely not matching
6. Walk away with a practical checklist for your own debugging, and a note on two settings that do not currently behave the way their names suggest

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer
from mbox.config import TableConfig, TableFieldConfig, IndexType

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


## 1. A perfect match on one field, gone entirely

`datasets/support_catalog.csv` is a small product catalog we'll use for the first four scenarios.

In [3]:
catalog = pd.read_csv("datasets/support_catalog.csv")
catalog

,product_id,product_name,description
0,B88-EXT,Extended Battery Pack,Rechargeable power cell for outdoor gear
1,A12-PWR,Portable Power Bank,Compact portable power bank for phones and tab...
2,C99-SNS,Motion Sensor Camera,Motion sensor camera with night vision for hom...
3,D45-REL,Smart Relay Switch,Smart relay switch for home automation systems


Here is a query where `product_name` is an exact, letter-for-letter match, but `description` is searched with something completely unrelated.

In [4]:
index = TableIndexer.create_index(catalog, index_columns=["product_id", "product_name", "description"], tmp_dir="tmp_index")

result = index.match(
    product_name="Extended Battery Pack",
    description="Quantum flux capacitor housing",
    include_field_scores=True
)
result

,query_row,index_row,product_name_candidate,description_candidate,product_id_candidate,overall_score,product_name_score,description_score
0,0,-1,,,,0,0,0


`index_row` is `-1`. Every score column reads `0`, including `product_name_score`, even though `"Extended Battery Pack"` is a letter-for-letter match against the indexed data. This is not a low-scoring match, it is no match at all.

A multi-field query needs enough relevant signal in *every* field you search, not just one of them. `description="Quantum flux capacitor housing"` shares essentially nothing with any indexed description, and that single field's failure took the entire candidate down with it, regardless of how perfect `product_name` was.

## 2. Diagnosing which field is the problem

When a multi-field query returns nothing, the fastest way to find out which field is responsible is to query each field on its own, one at a time, and see which one still finds a candidate.

In [5]:
name_only = index.match(product_name="Extended Battery Pack", include_field_scores=True)
print("product_name alone:")
display(name_only)

description_only = index.match(description="Quantum flux capacitor housing", include_field_scores=True)
print("\ndescription alone:")
display(description_only)

product_name alone:


,query_row,index_row,product_name_candidate,product_id_candidate,description_candidate,overall_score,product_name_score
0,0,0,Extended Battery Pack,B88-EXT,Rechargeable power cell for outdoor gear,100,100



description alone:


,query_row,index_row,description_candidate,product_id_candidate,product_name_candidate,overall_score,description_score
0,0,-1,,,,0,0


`product_name` alone finds a perfect match, `product_name_score = 100`. `description` alone finds nothing at all. That immediately tells you where the problem is: not in your `product_name` query, not in your index, specifically in the `description` value you searched with.

This is the general technique worth remembering: **when a combined query fails, split it apart.** Query each field independently, and whichever one comes back empty on its own is the one to investigate first.

## 3. A short field that ran out of room

Recall from `02-reading_field_level_scores.ipynb` that shorter fields tolerate far fewer edits before the match disappears entirely. `datasets/debug_id_queries.csv` holds a handful of typo'd versions of `product_id`, `"B88-EXT"`, to test that directly.

In [6]:
id_queries = pd.read_csv("datasets/debug_id_queries.csv")
id_queries

,query
0,B88-EXT
1,888-EXY
2,888-EXZ
3,888-EYZ


In [7]:
id_index = TableIndexer.create_index(pd.DataFrame({"product_id": ["B88-EXT"]}), index_columns=["product_id"], tmp_dir="tmp_index")

result = id_index.match(
    queries=id_queries.rename(columns={"query": "product_id"}),
    include_field_scores=True
)
id_queries.assign(found=result["index_row"] != -1, product_id_score=result["product_id_score"])

,query,found,product_id_score
0,B88-EXT,True,100
1,888-EXY,True,36
2,888-EXZ,True,36
3,888-EYZ,False,0


`"B88-EXT"` is only 7 characters long. Two substitutions still find a match, with a low score. A third substitution, `"888-EYZ"`, loses the match completely.

If a colleague reports "this product ID search used to work and now it doesn't," and the ID field is short, this is one of the first things worth checking: not a bug, but a genuine, expected consequence of how little room a short field has for `APPROX` to absorb noise.

## 4. An `IndexType` mismatch changing what "the same query" matches

`03-phrase_fields_explained.ipynb` and `04-term_fields_explained.ipynb` showed that `PHRASE` and `TERM` handle a reordered or shortened query very differently on identical data. If `product_name` was ever indexed as `TERM` instead of the `PHRASE` it gets inferred as here, by an explicit override, an inherited schema, or a config file from another service, queries that "should" work will start failing with no other change at all. `datasets/debug_name_reorder_queries.csv` holds three versions of the same query.

In [8]:
reorder_queries = pd.read_csv("datasets/debug_name_reorder_queries.csv")
reorder_queries

,label,query
0,exact,Extended Battery Pack
1,reordered,Battery Extended Pack
2,missing_word,Extended Battery


In [9]:
term_config = TableConfig(fields=[TableFieldConfig(column="product_name", index_type=IndexType.TERM)])
term_index = TableIndexer.create_index(df=pd.DataFrame({"product_name": catalog["product_name"]}), config_overrides=term_config, tmp_dir="tmp_index")

phrase_config = TableConfig(fields=[TableFieldConfig(column="product_name", index_type=IndexType.PHRASE)])
phrase_index = TableIndexer.create_index(df=pd.DataFrame({"product_name": catalog["product_name"]}), config_overrides=phrase_config, tmp_dir="tmp_index")

term_result = term_index.match(queries=reorder_queries[["query"]].rename(columns={"query": "product_name"}), include_field_scores=True)
phrase_result = phrase_index.match(queries=reorder_queries[["query"]].rename(columns={"query": "product_name"}), include_field_scores=True)

reorder_queries.assign(TERM_score=term_result["product_name_score"], PHRASE_score=phrase_result["product_name_score"])

config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.
config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.


,label,query,TERM_score,PHRASE_score
0,exact,Extended Battery Pack,100,100
1,reordered,Battery Extended Pack,0,100
2,missing_word,Extended Battery,64,96


Same query, same underlying text, two completely different outcomes depending on nothing but `IndexType`. The reordered query scores `100` under `PHRASE` and fails outright under `TERM`. If a field's behavior seems inconsistent with what this cookbook has told you to expect, `index.get_index_config().get_field("your_column").index_type`, the technique from `01-index_types_and_recall_modes.ipynb` and `02-reading_field_level_scores.ipynb`, is the first thing to check, before assuming anything else is wrong.

## 5. A candidate silently filtered by `minimum_quality`

This one is the trickiest to catch, because there is no `-1` and no obviously broken score, the candidate simply is not in your results, and the reason is a rule you set yourself, possibly a while ago, in a different notebook or script. `datasets/weighting_two_candidates.csv`, from `02-reading_field_level_scores.ipynb`, is built for exactly this: two products that both have a real claim to matching the same query.

In [10]:
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

candidates = pd.read_csv("datasets/weighting_two_candidates.csv")
candidates_index = TableIndexer.create_index(candidates, index_columns=["product_name", "description"], tmp_dir="tmp_index")

gated_config = TableRecallConfig(
    fields=[
        TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                minimum_quality=70, weight=50, mode=TableRecallMode.APPROX),
        TableRecallFieldConfig(input_column="description", indexed_column="description",
                                minimum_quality=0, weight=50, mode=TableRecallMode.APPROX)
    ],
    max_results=5,
    include_field_scores=True
)

query = pd.DataFrame({"product_name": ["Extended Battery Pack"], "description": ["Extended Battery Pack"]})
gated_results = candidates_index.match(queries=query, config=gated_config)

print("With minimum_quality=70 on product_name:")
gated_results

With minimum_quality=70 on product_name:


,query_row,index_row,product_name_candidate,description_candidate,overall_score,product_name_score,description_score
0,0,0,Extended Battery Pack,Compact accessory offering extended battery li...,64,100,29


Only one candidate comes back. Let's remove the `minimum_quality` gate and run the identical query, to see what was actually being filtered out.

In [11]:
ungated_config = TableRecallConfig(
    fields=[
        TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                minimum_quality=0, weight=50, mode=TableRecallMode.APPROX),
        TableRecallFieldConfig(input_column="description", indexed_column="description",
                                minimum_quality=0, weight=50, mode=TableRecallMode.APPROX)
    ],
    max_results=5,
    include_field_scores=True
)

ungated_results = candidates_index.match(queries=query, config=ungated_config)
print("Without the gate:")
ungated_results

Without the gate:


,query_row,index_row,product_name_candidate,description_candidate,overall_score,product_name_score,description_score
0,0,0,Extended Battery Pack,Compact accessory offering extended battery li...,64,100,29
1,0,1,Extended Cell Battery,Extended battery pack accessory for compact ca...,64,42,86


`"Extended Cell Battery"` was there all along, with a perfectly respectable `overall_score = 64`. The `minimum_quality=70` floor on `product_name` excluded it, because its `product_name_score` of `42` did not clear that bar, even though nothing else about the match was wrong.

This is why `04-recall-tuning/02-weighting_fields_for_better_precision.ipynb` calls `minimum_quality` a hard floor rather than a discount. If you inherit a `TableRecallConfig` from somewhere else, a shared file, an older notebook, a teammate's code, and a candidate you expect to see is missing, checking every field's `minimum_quality` setting should be high on your list, right alongside checking `index_row`.

## 6. A practical debugging checklist

When a query does not return what you expect, work through these in order:

1. **Check `index_row`.** If it is `-1`, no candidate cleared the threshold for at least one searched field. This is not a scoring problem, it is a candidacy problem.
2. **Isolate each field.** Query every field in your multi-field search on its own. Whichever one comes back empty independently is where the problem lives.
3. **Check field length.** If the problematic field is short, three or four characters, consider whether `APPROX` simply ran out of room, rather than assuming something is broken.
4. **Check the field's `IndexType`.** `index.get_index_config().get_field("column").index_type` tells you exactly what's being used. A `PHRASE`-shaped query run against a `TERM` or `IDENT` field will behave nothing like `03-phrase_fields_explained.ipynb` led you to expect.
5. **Check every `minimum_quality` setting.** If you are using a `TableRecallConfig`, especially one you did not write yourself moments ago, re-run the same query with every `minimum_quality` set to `0` and compare. A candidate that reappears was being filtered, not failing to match.

Every one of these is something you can check directly, by running one more query, rather than guessing. That is the entire point of explainability: a bad match is not a black box, it is something you can take apart.

## A note on two settings that don't currently do what their names suggest

You will see `min_total_match_value` passed as `0` throughout this cookbook, described as a floor being deliberately disabled "to keep low-scoring results visible for comparison." While preparing this notebook, that claim was tested properly for the first time, on the exact same `support_catalog.csv` used above, and it does not hold up.

A candidate scoring `42` should be excluded by the SDK's own documented default (`min_total_match_value = 50`, "minimum total match value required for a candidate to be returned"). It is not:

In [12]:
low_score_result = index.match(
    product_name="Extendd Battry",
    description="Rechargeable power for camping trip",
    include_field_scores=True
)
low_score_result[["index_row", "overall_score"]]

,index_row,overall_score
0,0,42


No `min_total_match_value` was even passed here, this is the SDK's own default. `overall_score` is `42`, well under the default floor of `50`, and the candidate is still returned. Setting `min_total_match_value` explicitly, to `0`, `50`, or `99`, changes nothing, it does not filter this candidate at any value tested.

`max_quality_spread`, the other candidate-list-level setting on `TableRecallConfig`, has the same problem. Querying `product_catalog.csv` for `"Battery"` returns two candidates scoring `89` and `88`. `max_quality_spread=0`, which should keep only candidates tied with the best score, still returns both.

**`minimum_quality`, the per-field setting used in Section 5, is not affected by this and works exactly as documented.** The issue is specific to the two candidate-list-level settings, `min_total_match_value` and `max_quality_spread`, both on `TableRecallConfig` rather than `TableRecallFieldConfig`. Until this is fixed upstream, do not rely on either one to actually exclude a low-scoring candidate, check `overall_score` yourself after the fact, or use `minimum_quality` on individual fields where a hard floor is required.

## Next steps

You now have the full explainability toolkit: how scores are actually calculated, what each `IndexType` does with them, how weighting combines fields, and how to diagnose a match that does not behave the way you expected, including knowing which settings to trust. From here:

- **`06-agentic-ai/`** - use M|BOX as a grounded, explainable tool inside LLM and agent workflows, where being able to justify a match is not just convenient, it is often the entire point

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*